# Notebook 1: Feature Engineering
**AdaptiveBeta — AI-Powered Portfolio Optimisation**  
**Author:** Kunal | M.Tech AI & ML, Symbiosis Institute of Technology, Pune

This notebook:
1. Mounts Google Drive and loads all raw data files
2. Aligns everything to the NIFTY50 trading calendar
3. Computes rolling OLS betas (30d, 60d, 120d) and beta volatility
4. Builds the shared market feature matrix (VIX, macro, flows, sectors)
5. Builds per-stock feature matrices and stacks into one CSV
6. Saves all features to Drive for Notebook 2

**Outputs saved to Drive:**
- `features/stacked_features.csv` — full cross-sectional feature matrix
- `features/beta{30,60,120}d.csv` — rolling beta panels
- `features/betavol_{30,60,120}d.csv` — beta volatility panels
- `features/target_betavol_20d_ahead.csv` — prediction target
- `features/market_features.csv` — shared market features

In [ ]:
# Install required libraries
!pip install -q PyPortfolioOpt riskfolio-lib pykalman hmmlearn xgboost shap \
    quantstats cvxpy scikit-learn pandas numpy matplotlib seaborn plotly openpyxl torch

In [ ]:
import sys
import os
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/AI_Finance_Project'

# Add src/ to path (if running from repo clone)
REPO = '/content/drive/MyDrive/AI_Finance_Project/repo'  # adjust if needed
if os.path.exists(REPO):
    sys.path.insert(0, REPO)

# Setup directories
for d in ['features', 'models', 'results']:
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)

print('Drive mounted. ROOT:', ROOT)
print('Directories ready.')

## 1.1 — Configuration & Tickers

In [ ]:
TICKERS = [
    'RELIANCE.NS','TCS.NS','HDFCBANK.NS','INFY.NS','ICICIBANK.NS',
    'HINDUNILVR.NS','ITC.NS','SBIN.NS','BHARTIARTL.NS','KOTAKBANK.NS',
    'LT.NS','AXISBANK.NS','ASIANPAINT.NS','MARUTI.NS','BAJFINANCE.NS',
    'HCLTECH.NS','SUNPHARMA.NS','TITAN.NS','ULTRACEMCO.NS','NESTLEIND.NS',
    'WIPRO.NS','POWERGRID.NS','NTPC.NS','ONGC.NS','TECHM.NS',
    'JSWSTEEL.NS','TATASTEEL.NS','M&M.NS','ADANIENT.NS','ADANIPORTS.NS',
    'COALINDIA.NS','BAJAJFINSV.NS','HDFCLIFE.NS','SBILIFE.NS','DRREDDY.NS',
    'DIVISLAB.NS','CIPLA.NS','EICHERMOT.NS','HEROMOTOCO.NS','APOLLOHOSP.NS',
    'BAJAJ-AUTO.NS','BRITANNIA.NS','GRASIM.NS','INDUSINDBK.NS','TATACONSUM.NS',
    'UPL.NS','BPCL.NS','IOC.NS','HINDALCO.NS',
]

BETA_WINDOWS = [30, 60, 120]
TARGET_HORIZON = 20  # predict betavol 20 trading days ahead

print(f'Universe: {len(TICKERS)} tickers')

## 1.2 — Load & Align All Data

In [ ]:
# Master trading calendar from NIFTY50
nifty = pd.read_csv(f'{ROOT}/raw_data/market/nifty50.csv', parse_dates=['date']).set_index('date').sort_index()
TRADING_DAYS = nifty.index
print(f'Trading calendar: {TRADING_DAYS[0].date()} → {TRADING_DAYS[-1].date()} ({len(TRADING_DAYS)} days)')

def align(df_or_series, cols=None):
    """Reindex to NIFTY50 trading calendar and forward-fill."""
    out = df_or_series.reindex(TRADING_DAYS).ffill()
    if cols is not None and isinstance(out, pd.DataFrame):
        out = out[cols]
    return out

def read_csv(path, date_col='date'):
    return pd.read_csv(path, parse_dates=[date_col]).set_index(date_col).sort_index()

# Stock prices (49 tickers)
prices = read_csv(f'{ROOT}/raw_data/stocks/all_stocks_prices.csv')
available_tickers = [t for t in TICKERS if t in prices.columns]
prices = align(prices[available_tickers])
print(f'Prices: {prices.shape}  ({len(available_tickers)} tickers)')

# Market data
vix = align(read_csv(f'{ROOT}/raw_data/market/india_vix.csv'), ['Close'])

# Macro
usdinr = align(read_csv(f'{ROOT}/raw_data/macro/usdinr.csv'), ['Close'])
crude = align(read_csv(f'{ROOT}/raw_data/macro/crude_oil.csv'), ['Close'])
crude['Close'] = crude['Close'].clip(lower=0)  # handle 2020-04-20 negative WTI

rfr_df = read_csv(f'{ROOT}/raw_data/macro/risk_free_rate_91d_daily.csv')
rfr_col = 'risk_free_rate' if 'risk_free_rate' in rfr_df.columns else rfr_df.columns[0]
rfr = align(rfr_df[[rfr_col]])

repo_df = read_csv(f'{ROOT}/raw_data/macro/repo_rate_daily.csv')
repo_col = 'repo_rate' if 'repo_rate' in repo_df.columns else repo_df.columns[0]
repo = align(repo_df[[repo_col]])

# Flows (monthly → daily forward-fill)
fii_df = read_csv(f'{ROOT}/raw_data/flows/fii_flows.csv')
fii_col = 'fii_net_investment' if 'fii_net_investment' in fii_df.columns else fii_df.columns[0]
fii = align(fii_df[[fii_col]])

dii_df = read_csv(f'{ROOT}/raw_data/flows/dii_flows.csv')
dii_col = 'dii_net_investment' if 'dii_net_investment' in dii_df.columns else dii_df.columns[0]
dii = align(dii_df[[dii_col]])

# Sector indices
bank = align(read_csv(f'{ROOT}/raw_data/sector/nifty_bank.csv'), ['Close'])
nifty_it = align(read_csv(f'{ROOT}/raw_data/sector/nifty_it.csv'), ['Close'])
nifty_fmcg = align(read_csv(f'{ROOT}/raw_data/sector/nifty_fmcg.csv'), ['Close'])

print('All data loaded and aligned to trading calendar.')

## 1.3 — Log Returns

In [ ]:
# Log returns: ln(P_t / P_{t-1})
stock_ret = np.log(prices / prices.shift(1))
market_ret = np.log(nifty['Close'] / nifty['Close'].shift(1))

# Align to common valid index
common_idx = stock_ret.dropna(how='all').index.intersection(market_ret.dropna().index)
stock_ret = stock_ret.loc[common_idx]
market_ret = market_ret.loc[common_idx]

# HDFCLIFE / SBILIFE: fill NaN returns (pre-IPO) with 0
for t in ['HDFCLIFE.NS', 'SBILIFE.NS']:
    if t in stock_ret.columns:
        stock_ret[t] = stock_ret[t].fillna(0)

print(f'Returns — stocks: {stock_ret.shape}, market: {market_ret.shape}')
print(f'Date range: {common_idx[0].date()} → {common_idx[-1].date()}')

## 1.4 — Rolling Beta & Beta Volatility

In [ ]:
def rolling_beta(s_ret, m_ret, window):
    """Rolling OLS beta: Cov(r_s, r_m) / Var(r_m)"""
    cov = s_ret.rolling(window).cov(m_ret)
    var = m_ret.rolling(window).var()
    return cov / var

print('Computing rolling betas (3 windows × 49 stocks)...')
betas = {}
for w in BETA_WINDOWS:
    betas[w] = stock_ret.apply(lambda s: rolling_beta(s, market_ret, w))
    print(f'  Beta {w}d — non-NaN rows: {betas[w].notna().any(axis=1).sum()}')

# Beta volatility: rolling std of beta (captures instability)
betavol = {}
for w in BETA_WINDOWS:
    betavol[w] = betas[w].rolling(w).std()
    print(f'  BetaVol {w}d — non-NaN rows: {betavol[w].notna().any(axis=1).sum()}')

# PRIMARY TARGET: 20-day forward beta volatility (60d base)
# Row t → betavol that will be observed at t+20
target = betavol[60].shift(-TARGET_HORIZON)
print(f'\nTarget (20d fwd betavol) — non-NaN rows: {target.notna().any(axis=1).sum()}')

## 1.5 — Market Feature Matrix

In [ ]:
features = pd.DataFrame(index=common_idx)

# --- VIX features ---
vix_s = vix['Close'].reindex(common_idx).ffill()
features['vix_level']   = vix_s
features['vix_5d_chg']  = vix_s.pct_change(5)
features['vix_20d_chg'] = vix_s.pct_change(20)
features['vix_zscore']  = (vix_s - vix_s.rolling(252).mean()) / vix_s.rolling(252).std()
features['vix_above_25'] = (vix_s > 25).astype(int)

# --- Macro features ---
usd = np.log(usdinr['Close'].reindex(common_idx).ffill())
features['usdinr_5d_chg']  = usd.diff(5)
features['usdinr_20d_vol'] = usd.diff(1).rolling(20).std()

crd = np.log(crude['Close'].clip(lower=1e-6).reindex(common_idx).ffill())
features['crude_5d_chg']  = crd.diff(5)
features['crude_20d_vol'] = crd.diff(1).rolling(20).std()

features['repo_rate'] = repo[repo_col].reindex(common_idx).ffill()
features['rfr']        = rfr[rfr_col].reindex(common_idx).ffill()
features['rate_spread'] = features['repo_rate'] - features['rfr']

# --- Market features ---
features['market_ret_5d']  = market_ret.rolling(5).sum()
features['market_ret_20d'] = market_ret.rolling(20).sum()
features['market_vol_20d'] = market_ret.rolling(20).std()
features['market_vol_60d'] = market_ret.rolling(60).std()

# --- Sector momentum ---
features['bank_mom_10d']  = np.log(bank['Close'].reindex(common_idx).ffill()).diff(10)
features['it_mom_10d']    = np.log(nifty_it['Close'].reindex(common_idx).ffill()).diff(10)
features['fmcg_mom_10d']  = np.log(nifty_fmcg['Close'].reindex(common_idx).ffill()).diff(10)
features['bank_vs_nifty'] = features['bank_mom_10d'] - features['market_ret_20d']

# --- Flow features ---
fii_s = fii[fii_col].reindex(common_idx).ffill()
dii_s = dii[dii_col].reindex(common_idx).ffill()
features['fii_net']       = fii_s
features['dii_net']       = dii_s
features['combined_flow'] = fii_s + dii_s
features['fii_dii_ratio'] = fii_s / (dii_s.abs() + 1)

print(f'Market features: {features.shape[1]} columns')
print(features.tail(3))

## 1.6 — Per-Stock Feature Matrices

In [ ]:
FEATURE_COLS = features.columns.tolist()
all_stocks_list = []
skipped = []

for i, ticker in enumerate(available_tickers, 1):
    df = features.copy()

    ret = stock_ret[ticker]

    # Per-stock beta features
    df['beta_30']   = betas[30][ticker]
    df['beta_60']   = betas[60][ticker]
    df['beta_120']  = betas[120][ticker]
    df['betavol_30']  = betavol[30][ticker]
    df['betavol_60']  = betavol[60][ticker]  # base for target
    df['betavol_120'] = betavol[120][ticker]
    df['beta_60_30_spread'] = betas[60][ticker] - betas[30][ticker]
    df['betavol_trend']     = betavol[60][ticker].diff(10)

    # Per-stock return features
    df['stock_ret_5d']   = ret.rolling(5).sum()
    df['stock_vol_20d']  = ret.rolling(20).std()
    df['stock_vs_mkt']   = df['stock_ret_5d'] - features['market_ret_5d']

    # Target
    df['target'] = target[ticker]

    # Drop rows where ANY feature or target is NaN
    before = len(df)
    df = df.dropna()
    after = len(df)

    if after < 200:
        skipped.append(ticker)
        print(f'  SKIP {ticker}: only {after} valid rows')
        continue

    tmp = df.copy()
    tmp['ticker'] = ticker
    all_stocks_list.append(tmp)

    if i % 10 == 0:
        print(f'  [{i}/{len(available_tickers)}] {ticker}: {after} rows (dropped {before-after} NaN rows)')

print(f'\nBuilt feature matrices for {len(all_stocks_list)} stocks')
if skipped:
    print(f'Skipped (insufficient data): {skipped}')

In [ ]:
# Stack all stocks into one DataFrame
stacked = pd.concat(all_stocks_list).sort_index()
print(f'Stacked feature matrix: {stacked.shape}')
print(f'Date range: {stacked.index[0].date()} → {stacked.index[-1].date()}')
print(f'Columns: {list(stacked.columns)}')
stacked.head(3)

## 1.7 — Save All Features to Drive

In [ ]:
# Stacked features (main input to models)
stacked.to_csv(f'{ROOT}/features/stacked_features.csv')
print('Saved: stacked_features.csv')

# Beta panels
for w in BETA_WINDOWS:
    betas[w].to_csv(f'{ROOT}/features/beta{w}d.csv')
    betavol[w].to_csv(f'{ROOT}/features/betavol_{w}d.csv')
    print(f'Saved: beta{w}d.csv, betavol_{w}d.csv')

# Target
target.to_csv(f'{ROOT}/features/target_betavol_20d_ahead.csv')
print('Saved: target_betavol_20d_ahead.csv')

# Market features
features.to_csv(f'{ROOT}/features/market_features.csv')
print('Saved: market_features.csv')

print('\n✅ All features saved to Drive.')

## 1.8 — Quick Visualisation

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Beta of RELIANCE across windows
ax = axes[0, 0]
for w, c in zip(BETA_WINDOWS, ['#1D9E75', '#7F77DD', '#EF9F27']):
    betas[w]['RELIANCE.NS'].plot(ax=ax, color=c, lw=1, label=f'Beta {w}d')
ax.axhline(1.0, color='white', ls='--', lw=0.7)
ax.set_title('RELIANCE.NS — Rolling Beta'); ax.legend()

# BetaVol of RELIANCE
ax = axes[0, 1]
for w, c in zip(BETA_WINDOWS, ['#1D9E75', '#7F77DD', '#EF9F27']):
    betavol[w]['RELIANCE.NS'].plot(ax=ax, color=c, lw=1, label=f'BetaVol {w}d')
ax.set_title('RELIANCE.NS — Beta Volatility'); ax.legend()

# India VIX
ax = axes[1, 0]
vix_s.plot(ax=ax, color='#D85A30', lw=1)
ax.axhline(25, color='yellow', ls='--', lw=0.8, label='VIX=25 threshold')
ax.set_title('India VIX'); ax.legend()

# NIFTY50 log returns
ax = axes[1, 1]
market_ret.plot(ax=ax, color='#888780', lw=0.5, alpha=0.7)
ax.set_title('NIFTY50 Log Returns')

plt.suptitle('Feature Engineering Overview', fontsize=14)
plt.tight_layout()
plt.savefig(f'{ROOT}/results/feature_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: feature_overview.png')